# Iris データセットの探索と分類モデルの訓練

このノートブックでは、Iris データセットを使った分類問題を TDD で実装します。

## 1. 環境設定とパスの確認

In [7]:
import java.io.File

// 現在のワーキングディレクトリを確認
val currentDir = File(".").absolutePath
println("現在のディレクトリ: $currentDir")

// データファイルのパスを自動検出
val possiblePaths = listOf(
    "src/main/resources/data/iris.csv",           // app/kotlin から実行
    "../src/main/resources/data/iris.csv",        // notebook から実行
    "app/kotlin/src/main/resources/data/iris.csv", // プロジェクトルートから実行
    "../../src/main/resources/data/iris.csv"      // さらに深い場所から実行
)

val dataPath = possiblePaths.firstOrNull { File(it).exists() }
    ?: error("iris.csv が見つかりません。現在のディレクトリ: $currentDir")

println("データファイル: $dataPath")
println("ファイル存在確認: ${File(dataPath).exists()}")

現在のディレクトリ: C:\Users\PC202411-1\IdeaProjects\case-study-game-dev\app\kotlin\notebook\.
データファイル: ../src/main/resources/data/iris.csv
ファイル存在確認: true


## 2. データの読み込みと概要確認

In [ ]:
// Smile ML ライブラリの依存関係を追加
@file:Repository("https://jitpack.io")
@file:DependsOn("com.github.haifengl:smile-core:3.0.2")
@file:DependsOn("com.github.haifengl:smile-kotlin:3.0.2")

println("依存関係の設定完了")

In [ ]:
import smile.classification.DecisionTree
import smile.data.DataFrame
import smile.data.formula.Formula
import smile.data.vector.DoubleVector
import smile.data.vector.IntVector
import java.io.Serializable
import java.util.Properties

/**
 * Iris データセットを分類する決定木モデル
 */
class IrisClassifier(val maxDepth: Int = 2) : Serializable {
    
    var model: DecisionTree? = null
        private set
    
    private var labelMapping: Map<Int, String> = emptyMap()
    
    init {
        require(maxDepth >= 1) { "maxDepth must be at least 1" }
    }
    
    /**
     * CSV ファイルからデータを読み込む
     */
    fun loadData(filePath: String): Pair<Array<DoubleArray>, Array<String>> {
        val file = java.io.File(filePath)
        require(file.exists()) { "File not found: $filePath" }
        
        val lines = file.readLines()
        require(lines.isNotEmpty()) { "Empty file: $filePath" }
        
        // ヘッダー行を解析（BOMを除去）
        val headerLine = lines[0].replace("\uFEFF", "").trim()
        val header = headerLine.split(",").map { it.trim() }
        val sepalLengthIdx = header.indexOf("sepal_length")
        val sepalWidthIdx = header.indexOf("sepal_width")
        val petalLengthIdx = header.indexOf("petal_length")
        val petalWidthIdx = header.indexOf("petal_width")
        val speciesIdx = header.indexOf("species")
        
        require(sepalLengthIdx >= 0 && sepalWidthIdx >= 0 && petalLengthIdx >= 0 &&
                petalWidthIdx >= 0 && speciesIdx >= 0) {
            "Required columns not found in CSV"
        }
        
        // データ行を読み込み（欠損値を含む行は除外）
        val validRows = mutableListOf<Pair<DoubleArray, String>>()
        
        for (i in 1 until lines.size) {
            val values = lines[i].split(",")
            if (values.size != header.size) continue
            
            try {
                // 全ての値が空でないか確認
                if (values[sepalLengthIdx].isBlank() || values[sepalWidthIdx].isBlank() ||
                    values[petalLengthIdx].isBlank() || values[petalWidthIdx].isBlank() ||
                    values[speciesIdx].isBlank()) {
                    continue
                }
                
                val features = doubleArrayOf(
                    values[sepalLengthIdx].toDouble(),
                    values[sepalWidthIdx].toDouble(),
                    values[petalLengthIdx].toDouble(),
                    values[petalWidthIdx].toDouble()
                )
                val label = values[speciesIdx].trim()
                
                validRows.add(Pair(features, label))
            } catch (e: NumberFormatException) {
                continue
            }
        }
        
        require(validRows.isNotEmpty()) { "No valid data found in CSV" }
        
        val X = validRows.map { it.first }.toTypedArray()
        val y = validRows.map { it.second }.toTypedArray()
        
        return Pair(X, y)
    }
    
    /**
     * モデルを訓練する
     */
    fun train(X: Array<DoubleArray>, y: Array<String>) {
        require(X.isNotEmpty() && y.isNotEmpty()) { "Training data cannot be empty" }
        require(X.size == y.size) {
            "X and y must have the same length: ${X.size} != ${y.size}"
        }
        
        // ラベルを整数にマッピング
        val uniqueLabels = y.distinct().sorted()
        val labelToInt = uniqueLabels.withIndex().associate { it.value to it.index }
        labelMapping = labelToInt.entries.associate { it.value to it.key }
        val yInt = y.map { labelToInt[it]!! }.toIntArray()
        
        // DataFrame を作成
        val data = DataFrame.of(
            DoubleVector.of("sepal_length", X.map { it[0] }.toDoubleArray()),
            DoubleVector.of("sepal_width", X.map { it[1] }.toDoubleArray()),
            DoubleVector.of("petal_length", X.map { it[2] }.toDoubleArray()),
            DoubleVector.of("petal_width", X.map { it[3] }.toDoubleArray()),
            IntVector.of("species", yInt)
        )
        
        // モデルの訓練
        val formula = Formula.lhs("species")
        val props = Properties()
        props.setProperty("smile.decision_tree.max_depth", maxDepth.toString())
        model = DecisionTree.fit(formula, data, props)
    }
    
    /**
     * 予測を実行する
     */
    fun predict(X: Array<DoubleArray>): Array<String> {
        requireNotNull(model) { "Model has not been trained yet" }
        
        val dummySpecies = IntArray(X.size) { 0 }
        val testData = DataFrame.of(
            DoubleVector.of("sepal_length", X.map { it[0] }.toDoubleArray()),
            DoubleVector.of("sepal_width", X.map { it[1] }.toDoubleArray()),
            DoubleVector.of("petal_length", X.map { it[2] }.toDoubleArray()),
            DoubleVector.of("petal_width", X.map { it[3] }.toDoubleArray()),
            IntVector.of("species", dummySpecies)
        )
        
        val predictions = model!!.predict(testData)
        
        return predictions.map { prediction ->
            labelMapping[prediction] ?: error("Unknown prediction: $prediction")
        }.toTypedArray()
    }
    
    /**
     * モデルの性能を評価する
     */
    fun evaluate(X: Array<DoubleArray>, y: Array<String>): Double {
        requireNotNull(model) { "Model has not been trained yet" }
        
        val predictions = predict(X)
        val correct = predictions.zip(y).count { (pred, actual) -> pred == actual }
        return correct.toDouble() / y.size
    }
    
    /**
     * 訓練済みモデルをファイルに保存する
     */
    fun saveModel(filePath: String) {
        requireNotNull(model) { "No trained model to save" }
        
        java.io.ObjectOutputStream(java.io.FileOutputStream(filePath)).use { oos ->
            oos.writeObject(model)
        }
    }
    
    /**
     * 保存されたモデルをファイルから読み込む
     */
    fun loadModel(filePath: String) {
        java.io.ObjectInputStream(java.io.FileInputStream(filePath)).use { ois ->
            @Suppress("UNCHECKED_CAST")
            model = ois.readObject() as DecisionTree
        }
    }
}

// モデルの作成
val classifier = IrisClassifier(maxDepth = 3)

// データの読み込み
val (X, y) = classifier.loadData(dataPath)

println("=".repeat(60))
println("データの概要")
println("=".repeat(60))
println("サンプル数: ${X.size}")
println("特徴量数: ${X[0].size}")
println("クラス数: ${y.distinct().size}")
println()

// データの最初の5行を表示
println("最初の5サンプル:")
println("sepal_length, sepal_width, petal_length, petal_width, species")
for (i in 0 until minOf(5, X.size)) {
    println("${X[i][0]}, ${X[i][1]}, ${X[i][2]}, ${X[i][3]}, ${y[i]}")
}

## 3. Lets-Plot の初期化

In [ ]:
%use kandy

## 4. データの可視化

### 4.1 特徴量の分布（ヒストグラム）- sepal_length

In [ ]:
// sepal_length の分布をヒストグラムで可視化
val data1 = mapOf(
    "value" to X.map { it[0] },
    "species" to y.toList()
)

plot(data1) {
    histogram("value", bins = 20) {
        fillColor("species")
        alpha = 0.6
    }
    layout {
        title = "sepal_length の分布"
        xAxis.name = "sepal_length"
        yAxis.name = "度数"
    }
}

### 4.2 特徴量の分布（ヒストグラム）- petal_length

In [ ]:
// petal_length の分布をヒストグラムで可視化
val data2 = mapOf(
    "value" to X.map { it[2] },
    "species" to y.toList()
)

plot(data2) {
    histogram("value", bins = 20) {
        fillColor("species")
        alpha = 0.6
    }
    layout {
        title = "petal_length の分布"
        xAxis.name = "petal_length"
        yAxis.name = "度数"
    }
}

### 4.3 散布図 - 花びらの長さ vs 幅

In [ ]:
// 花びらの長さ vs 幅の散布図
val petalData = mapOf(
    "petal_length" to X.map { it[2] },
    "petal_width" to X.map { it[3] },
    "species" to y.toList()
)

plot(petalData) {
    points {
        x("petal_length")
        y("petal_width")
        color("species")
        size = 3.0
        alpha = 0.7
    }
    layout {
        title = "花びらの長さ vs 幅"
        xAxis.name = "petal_length (cm)"
        yAxis.name = "petal_width (cm)"
    }
}

### 4.4 散布図 - がく片の長さ vs 幅

In [ ]:
// がく片の長さ vs 幅の散布図
val sepalData = mapOf(
    "sepal_length" to X.map { it[0] },
    "sepal_width" to X.map { it[1] },
    "species" to y.toList()
)

plot(sepalData) {
    points {
        x("sepal_length")
        y("sepal_width")
        color("species")
        size = 3.0
        alpha = 0.7
    }
    layout {
        title = "がく片の長さ vs 幅"
        xAxis.name = "sepal_length (cm)"
        yAxis.name = "sepal_width (cm)"
    }
}

## 5. 基本統計量の確認

In [8]:
val featureNames = listOf("sepal_length", "sepal_width", "petal_length", "petal_width")

println("基本統計量:")
println("-".repeat(60))

featureNames.forEachIndexed { idx, name ->
    val values = X.map { it[idx] }
    val min = values.minOrNull() ?: 0.0
    val max = values.maxOrNull() ?: 0.0
    val mean = values.average()
    val sorted = values.sorted()
    val median = if (sorted.size % 2 == 0) {
        (sorted[sorted.size / 2 - 1] + sorted[sorted.size / 2]) / 2.0
    } else {
        sorted[sorted.size / 2]
    }
    
    println("$name:")
    println("  最小値: %.2f".format(min))
    println("  最大値: %.2f".format(max))
    println("  平均値: %.2f".format(mean))
    println("  中央値: %.2f".format(median))
    println()
}

org.jetbrains.kotlinx.jupyter.exceptions.ReplCompilerException: at Cell In[8], line 7, column 18: Unresolved reference: X
at Cell In[8], line 7, column 26: Unresolved reference: it

## 6. クラスの分布

In [9]:
println("クラスの分布:")
println("-".repeat(60))

val total = y.size.toDouble()
val classCounts = y.groupBy { it }.mapValues { it.value.size }

classCounts.toSortedMap().forEach { (species, count) ->
    val percentage = (count / total * 100)
    println("$species: $count 件 (%.1f%%)".format(percentage))
}
println()

org.jetbrains.kotlinx.jupyter.exceptions.ReplCompilerException: at Cell In[9], line 4, column 13: Unresolved reference: y
at Cell In[9], line 5, column 19: Unresolved reference: y
at Cell In[9], line 5, column 31: Unresolved reference: it
at Cell In[9], line 5, column 48: Unresolved reference: it
at Cell In[9], line 7, column 27: Overload resolution ambiguity: 
public inline fun <T> Iterable<TypeVariable(T)>.forEach(action: (TypeVariable(T)) -> Unit): Unit defined in kotlin.collections
public inline fun <K, V> Map<out TypeVariable(K), TypeVariable(V)>.forEach(action: (Map.Entry<TypeVariable(K), TypeVariable(V)>) -> Unit): Unit defined in kotlin.collections
at Cell In[9], line 7, column 37: Cannot infer a type for this parameter. Please specify it explicitly.

### クラス分布の棒グラフ

In [ ]:
// クラス分布の棒グラフ
val barData = mapOf(
    "species" to classCounts.keys.toList(),
    "count" to classCounts.values.toList()
)

plot(barData) {
    bars {
        x("species")
        y("count")
        fillColor("species")
        alpha = 0.8
    }
    layout {
        title = "クラスの分布"
        xAxis.name = "Species"
        yAxis.name = "サンプル数"
    }
}

## 7. モデルの訓練

In [80]:
println("モデルの訓練中...")
val startTime = System.currentTimeMillis()
classifier.train(X, y)
val trainingTime = System.currentTimeMillis() - startTime

println("訓練完了！")
println("訓練時間: ${trainingTime}ms")
println()

モデルの訓練中...
訓練完了！
訓練時間: 1ms



## 8. モデルの評価

In [81]:
val trainAccuracy = classifier.evaluate(X, y)
println("=".repeat(60))
println("モデルの評価結果")
println("=".repeat(60))
println("訓練正解率: %.2f%%".format(trainAccuracy * 100))
println()

モデルの評価結果
訓練正解率: 95.10%



## 9. 混同行列

In [82]:
println("混同行列:")
val predictions = classifier.predict(X)
val species = y.distinct().sorted()

println("実際 \\ 予測 | " + species.joinToString(" | "))
println("-".repeat(60))

// 混同行列のデータを準備
val confusionMatrix = mutableListOf<Triple<String, String, Int>>()

species.forEach { actualSpecies ->
    val actualIndices = y.indices.filter { y[it] == actualSpecies }
    print("%-15s | ".format(actualSpecies))
    species.forEach { predSpecies ->
        val count = actualIndices.count { predictions[it] == predSpecies }
        print("%3d | ".format(count))
        confusionMatrix.add(Triple(actualSpecies, predSpecies, count))
    }
    println()
}
println()

混同行列:
実際 \ 予測 | Iris-setosa | Iris-versicolor | Iris-virginica
------------------------------------------------------------
Iris-setosa     |  50 |   0 |   0 | 
Iris-versicolor |   0 |  47 |   1 | 
Iris-virginica  |   0 |   6 |  39 | 



### 混同行列のヒートマップ

In [ ]:
// 混同行列のヒートマップ
val heatmapData = mapOf(
    "actual" to confusionMatrix.map { it.first },
    "predicted" to confusionMatrix.map { it.second },
    "count" to confusionMatrix.map { it.third }
)

plot(heatmapData) {
    tiles {
        x("predicted")
        y("actual")
        fillColor("count")
        alpha = 0.9
    }
    layout {
        title = "混同行列"
        xAxis.name = "予測されたクラス"
        yAxis.name = "実際のクラス"
    }
}

## 10. クラス別の性能

In [84]:
println("クラス別の性能:")
println("-".repeat(60))

val classAccuracies = mutableListOf<Pair<String, Double>>()

species.forEach { targetSpecies ->
    val indices = y.indices.filter { y[it] == targetSpecies }
    val correct = indices.count { predictions[it] == targetSpecies }
    val total = indices.size
    val classAccuracy = correct.toDouble() / total
    classAccuracies.add(Pair(targetSpecies, classAccuracy))

    println("$targetSpecies:")
    println("  正解率: %.2f%% ($correct / $total)".format(classAccuracy * 100))
}
println()

クラス別の性能:
------------------------------------------------------------
Iris-setosa:
  正解率: 100.00% (50 / 50)
Iris-versicolor:
  正解率: 97.92% (47 / 48)
Iris-virginica:
  正解率: 86.67% (39 / 45)



### クラス別正解率の棒グラフ

In [ ]:
// クラス別正解率の棒グラフ
val accuracyData = mapOf(
    "species" to classAccuracies.map { it.first },
    "accuracy" to classAccuracies.map { it.second * 100 }
)

plot(accuracyData) {
    bars {
        x("species")
        y("accuracy")
        fillColor("species")
        alpha = 0.8
    }
    layout {
        title = "クラス別の正解率"
        xAxis.name = "Species"
        yAxis.name = "正解率 (%)"
    }
}

## 11. 個別予測の例

In [86]:
println("個別予測の例:")
println("-".repeat(60))

val testSamples = listOf(
    Triple(doubleArrayOf(5.1, 3.5, 1.4, 0.2), "setosa", "setosa の典型的な特徴"),
    Triple(doubleArrayOf(6.5, 3.0, 5.2, 2.0), "virginica", "virginica の典型的な特徴"),
    Triple(doubleArrayOf(5.7, 2.8, 4.1, 1.3), "versicolor", "versicolor の典型的な特徴")
)

testSamples.forEachIndexed { i, (sample, expected, description) ->
    val prediction = classifier.predict(arrayOf(sample))[0]
    val isCorrect = prediction == expected
    val mark = if (isCorrect) "✓" else "✗"
    println("$mark サンプル ${i+1} ($description):")
    println("  特徴量: [${sample.joinToString(", ")}]")
    println("  予測: $prediction (期待: $expected)")
    println()
}

個別予測の例:
------------------------------------------------------------
✗ サンプル 1 (setosa の典型的な特徴):
  特徴量: [5.1, 3.5, 1.4, 0.2]
  予測: Iris-setosa (期待: setosa)

✗ サンプル 2 (virginica の典型的な特徴):
  特徴量: [6.5, 3.0, 5.2, 2.0]
  予測: Iris-virginica (期待: virginica)

✗ サンプル 3 (versicolor の典型的な特徴):
  特徴量: [5.7, 2.8, 4.1, 1.3]
  予測: Iris-virginica (期待: versicolor)



### 予測結果の可視化

In [ ]:
// 元のデータと新しいサンプルを一緒にプロット
val allPetalLength = X.map { it[2] }.toMutableList()
val allPetalWidth = X.map { it[3] }.toMutableList()
val allSpecies = y.toMutableList()
val allType = MutableList(X.size) { "元データ" }

// 新しいサンプルを追加
testSamples.forEach { (sample, expected, _) ->
    allPetalLength.add(sample[2])
    allPetalWidth.add(sample[3])
    val prediction = classifier.predict(arrayOf(sample))[0]
    allSpecies.add(prediction)
    allType.add("予測")
}

val combinedData = mapOf(
    "petal_length" to allPetalLength,
    "petal_width" to allPetalWidth,
    "species" to allSpecies,
    "type" to allType
)

plot(combinedData) {
    points {
        x("petal_length")
        y("petal_width")
        color("species")
        symbol("type")
        alpha = 0.7
    }
    layout {
        title = "予測結果の可視化"
        xAxis.name = "petal_length (cm)"
        yAxis.name = "petal_width (cm)"
    }
}

## 12. モデルの保存

In [88]:
import java.io.File

// モデル保存先のパスを構築
val modelDir = when {
    File("model").exists() || File(".").resolve("model").parentFile.exists() -> "model"
    File("../model").parentFile.exists() -> "../model"
    File("app/kotlin/model").parentFile.exists() -> "app/kotlin/model"
    else -> "model" // デフォルト
}

val modelPath = "$modelDir/iris_model.ser"
File(modelPath).parentFile?.mkdirs()

classifier.saveModel(modelPath)

println("=".repeat(60))
println("モデルの保存完了")
println("=".repeat(60))
println("保存先: $modelPath")
println("ファイルサイズ: ${File(modelPath).length()} bytes")

モデルの保存完了
保存先: model/iris_model.ser
ファイルサイズ: 2142 bytes


## まとめ

このノートブックでは、Iris データセットを使った分類問題を TDD で実装しました。

### 実施内容

1. **データの読み込みと探索**: CSV からのデータ読み込み、欠損値の処理
2. **データの可視化**: ヒストグラム、散布図、棒グラフによる分布確認
3. **モデルの訓練**: 決定木モデルの訓練（max_depth=3）
4. **モデルの評価**: 正解率の計算、混同行列の作成
5. **予測の実行**: 新しいデータでの分類予測と可視化
6. **モデルの保存**: モデルの永続化

### 次のステップ

- データを訓練用とテスト用に分割して過学習をチェック
- クロスバリデーションで性能を評価
- 他のデータセット（Cinema、Boston、Survived）に挑戦
- Web API 化（Ktor による REST API 実装）

---

お疲れ様でした！TDD による機械学習開発の基礎を習得しました！